# 03. Baseline CNN (직접 설계)

Conv-BN-ReLU-MaxPool × 3 → GAP → Linear(64) → Dropout(0.5) → 6 class, 클래스 가중 CrossEntropy, 5 epoch.
테스트 예측 확률은 [`results/pred_baseline_cnn.csv`](../results/pred_baseline_cnn.csv)


In [17]:
# %%
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# %%
import os, time
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# %% [markdown]
# ### [Step 1] 압축 해제 및 경로 설정
import zipfile, os

zip_path = "/content/drive/MyDrive/images.zip"
IMG_DIR  = "/content/images"

print("압축 해제 중...")
with zipfile.ZipFile(zip_path, 'r') as zf:
    total = len(zf.namelist())
    for i, file in enumerate(zf.namelist()):
        zf.extract(file, "/content/")
        if (i + 1) % 5000 == 0:
            print(f"  {i+1}/{total} ({100*(i+1)/total:.0f}%)")
print(f"완료! {len(os.listdir(IMG_DIR))}개")
IMG_DIR = "/content/images"

# %% [markdown]
# ### [Step 2] Dataset
STYLE_CLASSES = ['Casual', 'Formal', 'Romantic', 'Sporty', 'Street', 'Vintage']

class FashionStyleDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        df = pd.read_csv(csv_file)
        all_files = set(os.listdir(img_dir))
        self.df = df[df['id'].apply(lambda x: f"{x}.jpg" in all_files)].reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.class_to_idx = {c: i for i, c in enumerate(STYLE_CLASSES)}
        print(f"{csv_file}: {len(self.df)}개 로드")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, f"{row['id']}.jpg")
        try:
            img = Image.open(img_path).convert("RGB")
        except Exception:
            img = Image.new("RGB", (224, 224), (128, 128, 128))
        if self.transform:
            img = self.transform(img)
        label = self.class_to_idx.get(row.get('style_label', 'Casual'), 0)
        return img, label, int(row['id'])

# %% [markdown]
# ### [Step 3] DataLoader
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

train_ds = FashionStyleDataset("train (1).csv", IMG_DIR, train_tf)
val_ds   = FashionStyleDataset("val (1).csv",   IMG_DIR, val_tf)
test_ds  = FashionStyleDataset("test (1).csv",  IMG_DIR, val_tf)

print("\n클래스 분포:")
print(train_ds.df['style_label'].value_counts())

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

# %% [markdown]
# ### [Step 4] 모델 + 클래스 가중치
class BaselineCNN(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128, 3, padding=1), nn.BatchNorm2d(128),nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1,1)), nn.Flatten(),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

model = BaselineCNN().to(device)

counts  = train_ds.df['style_label'].value_counts()
weights = torch.tensor([1.0/counts[c] for c in STYLE_CLASSES], dtype=torch.float32).to(device)
weights = weights / weights.sum() * len(STYLE_CLASSES)
print("\n클래스 가중치:")
for c, w in zip(STYLE_CLASSES, weights.cpu()): print(f"  {c:10s}: {w:.4f}")

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.5)

# %% [markdown]
# ### [Step 5] 학습
from tqdm import tqdm
import time

EPOCHS = 5
best_val_acc, best_state = 0.0, None

print("\n학습 시작!")
for epoch in range(EPOCHS):
    model.train()
    correct, total, t0 = 0, 0, time.time()

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:02d}/{EPOCHS} [Train]")
    for imgs, labels, _ in pbar:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        correct += outputs.argmax(1).eq(labels).sum().item()
        total   += len(labels)
        pbar.set_postfix(acc=f"{100.*correct/total:.1f}%")

    model.eval()
    vc, vt = 0, 0
    with torch.no_grad():
        for imgs, labels, _ in tqdm(val_loader, desc=f"Epoch {epoch+1:02d}/{EPOCHS} [Val]  "):
            imgs, labels = imgs.to(device), labels.to(device)
            vc += model(imgs).argmax(1).eq(labels).sum().item()
            vt += len(labels)

    scheduler.step()
    marker = " ← best" if vc/vt > best_val_acc else ""
    print(f"  [Epoch {epoch+1:02d}/{EPOCHS}] Train Acc: {100.*correct/total:.2f}%  Val Acc: {100.*vc/vt:.2f}%  ({time.time()-t0:.0f}초){marker}\n")

    if vc/vt > best_val_acc:
        best_val_acc = vc/vt
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

print(f"\n완료! Best Val Acc: {100.*best_val_acc:.2f}%")
model.load_state_dict(best_state)
torch.save(model.state_dict(), "baseline_cnn.pth")

# %% [markdown]
# ### [Step 6] 테스트 추론 및 저장
model.eval()
all_ids, all_probs = [], []
with torch.no_grad():
    for imgs, _, ids in test_loader:
        probs = torch.softmax(model(imgs.to(device)), dim=1).cpu().numpy()
        all_ids.extend(ids.tolist())
        all_probs.append(probs)

pred_df = pd.DataFrame(np.vstack(all_probs), columns=[f"prob_{c}" for c in STYLE_CLASSES])
pred_df.insert(0, 'id', all_ids)
os.makedirs('outputs', exist_ok=True)
pred_df.to_csv('outputs/pred_baseline_cnn.csv', index=False)
print("저장 완료!")

# %% [markdown]
# ### [Step 7] 클래스별 정확도
test_df = pd.read_csv("test (1).csv")
merged  = pred_df.merge(test_df[['id','style_label']], on='id')
merged['pred'] = merged[[f"prob_{c}" for c in STYLE_CLASSES]].idxmax(axis=1).str.replace('prob_','')
print(f"\n전체 정확도: {(merged['pred']==merged['style_label']).mean():.4f}")
for c in STYLE_CLASSES:
    sub = merged[merged['style_label']==c]
    if len(sub)==0: continue
    print(f"  {c:10s}: {(sub['pred']==sub['style_label']).mean():.3f}  (n={len(sub)})")


Mounted at /content/drive
Using device: cuda
압축 해제 중...
  5000/88884 (6%)
  10000/88884 (11%)
  15000/88884 (17%)
  20000/88884 (23%)
  25000/88884 (28%)
  30000/88884 (34%)
  35000/88884 (39%)
  40000/88884 (45%)
  45000/88884 (51%)
  50000/88884 (56%)
  55000/88884 (62%)
  60000/88884 (68%)
  65000/88884 (73%)
  70000/88884 (79%)
  75000/88884 (84%)
  80000/88884 (90%)
  85000/88884 (96%)
완료! 44441개
train (1).csv: 21398개 로드
val (1).csv: 3058개 로드
test (1).csv: 6114개 로드

클래스 분포:
style_label
Casual      15096
Sporty       2538
Vintage      1733
Formal       1278
Street        383
Romantic      370
Name: count, dtype: int64

클래스 가중치:
  Casual    : 0.0557
  Formal    : 0.6581
  Romantic  : 2.2733
  Sporty    : 0.3314
  Street    : 2.1961
  Vintage   : 0.4853

학습 시작!


Epoch 01/5 [Val]  : 100%|██████████| 24/24 [01:57<00:00,  4.88s/it]


  [Epoch 01/5] Train Acc: 13.53%  Val Acc: 19.23%  (982초) ← best



Epoch 02/5 [Val]  : 100%|██████████| 24/24 [01:57<00:00,  4.88s/it]


  [Epoch 02/5] Train Acc: 23.70%  Val Acc: 28.88%  (946초) ← best



Epoch 03/5 [Val]  : 100%|██████████| 24/24 [01:59<00:00,  4.98s/it]


  [Epoch 03/5] Train Acc: 26.64%  Val Acc: 33.00%  (962초) ← best



Epoch 04/5 [Val]  : 100%|██████████| 24/24 [01:57<00:00,  4.90s/it]


  [Epoch 04/5] Train Acc: 29.57%  Val Acc: 28.35%  (969초)



Epoch 05/5 [Val]  : 100%|██████████| 24/24 [01:52<00:00,  4.67s/it]


  [Epoch 05/5] Train Acc: 29.91%  Val Acc: 33.36%  (980초) ← best


완료! Best Val Acc: 33.36%
저장 완료!

전체 정확도: 0.3327
  Casual    : 0.222  (n=4314)
  Formal    : 0.674  (n=365)
  Romantic  : 0.613  (n=106)
  Sporty    : 0.552  (n=725)
  Street    : 0.697  (n=109)
  Vintage   : 0.588  (n=495)
